# 🛰️ AgroSat — Monitoramento Agrícola com Dados de Satélite
## Global Solutions 2026 | FIAP

**Disciplina:** Data Science, Inovação e Tecnologia  
**Tema:** Economia Espacial a serviço da Agricultura Brasileira  
**Dataset:** Dados simulados com base na metodologia do INPE/BDQueimadas e NASA EarthData  

---

### 👥 Integrantes do Grupo
| Nome | RM |
|------|-----|
| Aluno 1 (Responsável) | RM XXXXX |
| Aluno 2 | RM XXXXX |
| Aluno 3 | RM XXXXX |

---

### 📋 Descrição do Problema

**Cenário:** O Brasil é um dos maiores produtores agrícolas do mundo, mas pequenos e médios produtores (que representam ~77% dos estabelecimentos rurais) tomam decisões sem acesso a dados climáticos e ambientais precisos. Queimadas descontroladas, secas e desmatamento afetam diretamente a produtividade e o meio ambiente.

**Problema central:** Como dados de satélite sobre focos de queimada, precipitação e índice de vegetação (NDVI) podem ajudar na tomada de decisão agrícola e ambiental no Brasil?

**Relevância:** Segundo o INPE, o Brasil registrou mais de 270.000 focos de queimada só em 2020. Esses eventos destroem lavouras, prejudicam a qualidade do ar e aceleram mudanças climáticas — impactando diretamente a segurança alimentar.

**Quem é impactado:** Pequenos agricultores, cooperativas rurais, órgãos ambientais e prefeituras municipais.

**Como os dados ajudam:** Analisando padrões históricos de queimadas, precipitação e vegetação, é possível prever períodos de risco, orientar o plantio e direcionar ações de fiscalização ambiental.

---

### 🔗 Alinhamento com ODS da ONU
- **ODS 2** — Fome Zero e Agricultura Sustentável
- **ODS 13** — Ação Contra a Mudança Global do Clima
- **ODS 15** — Vida Terrestre (proteção dos ecossistemas)


## 1. 📦 Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, ttest_ind, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
sns.set_palette('viridis')
sns.set_style('whitegrid')

print("✅ Bibliotecas importadas com sucesso!")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")
print(f"   seaborn {sns.__version__}")


## 2. 📂 Carregamento e Descrição dos Dados

In [ ]:
# Carregamento das bases de dados
df_queimadas = pd.read_csv('queimadas_brasil_2015_2024.csv')
df_prec      = pd.read_csv('precipitacao_brasil_2015_2024.csv')
df_ndvi      = pd.read_csv('ndvi_brasil_2015_2024.csv')

print("=" * 55)
print("📊 RESUMO DAS BASES CARREGADAS")
print("=" * 55)
for nome, df in [('Queimadas', df_queimadas),
                 ('Precipitação', df_prec),
                 ('NDVI', df_ndvi)]:
    print(f"\n🗂️  {nome}")
    print(f"   Registros : {len(df):,}")
    print(f"   Colunas   : {list(df.columns)}")


In [ ]:
# Visualização das primeiras linhas de cada base
print("\n📋 QUEIMADAS — primeiras linhas:")
display(df_queimadas.head())

print("\n📋 PRECIPITAÇÃO — primeiras linhas:")
display(df_prec.head())

print("\n📋 NDVI — primeiras linhas:")
display(df_ndvi.head())


## 3. 🧹 Limpeza e Tratamento dos Dados
> **Etapa obrigatória.** Garantimos qualidade antes de qualquer análise.


In [ ]:
print("=" * 55)
print("🔍 DIAGNÓSTICO DE QUALIDADE DOS DADOS")
print("=" * 55)

for nome, df in [('Queimadas', df_queimadas),
                 ('Precipitação', df_prec),
                 ('NDVI', df_ndvi)]:
    nulos = df.isnull().sum().sum()
    dupl  = df.duplicated().sum()
    print(f"\n{nome}:")
    print(f"  Valores nulos  : {nulos}")
    print(f"  Duplicatas     : {dupl}")
    print(f"  Tipos:\n{df.dtypes.to_string()}")


In [ ]:
# ── Queimadas: remover outliers extremos via IQR ──
Q1 = df_queimadas['focos'].quantile(0.25)
Q3 = df_queimadas['focos'].quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

antes = len(df_queimadas)
df_queimadas_limpo = df_queimadas[
    (df_queimadas['focos'] >= limite_inf) &
    (df_queimadas['focos'] <= limite_sup)
].copy()
depois = len(df_queimadas_limpo)

print(f"Outliers removidos (Queimadas): {antes - depois} registros")
print(f"  Limite inferior : {limite_inf:.1f}")
print(f"  Limite superior : {limite_sup:.1f}")

# ── Precipitação: substituir negativos por 0 ──
neg = (df_prec['precipitacao_mm'] < 0).sum()
df_prec_limpo = df_prec.copy()
df_prec_limpo['precipitacao_mm'] = df_prec_limpo['precipitacao_mm'].clip(lower=0)
print(f"\nValores negativos de precipitação corrigidos: {neg}")

# ── NDVI: garantir range [0, 1] ──
fora = ((df_ndvi['ndvi_medio'] < 0) | (df_ndvi['ndvi_medio'] > 1)).sum()
df_ndvi_limpo = df_ndvi.copy()
df_ndvi_limpo['ndvi_medio'] = df_ndvi_limpo['ndvi_medio'].clip(0, 1)
print(f"Valores NDVI fora do range [0,1] corrigidos: {fora}")

print("\n✅ Limpeza concluída com sucesso!")


In [ ]:
# Normalização: criar coluna de focos por 1000 (escala)
df_queimadas_limpo['focos_norm'] = (
    (df_queimadas_limpo['focos'] - df_queimadas_limpo['focos'].min()) /
    (df_queimadas_limpo['focos'].max() - df_queimadas_limpo['focos'].min())
)

# Criar chave de merge (ano + mes + estado)
df_merged = df_queimadas_limpo.merge(
    df_prec_limpo[['ano', 'mes', 'estado', 'precipitacao_mm']],
    on=['ano', 'mes', 'estado'], how='inner'
)
print(f"Base integrada (queimadas + precipitação): {len(df_merged):,} registros")
print(df_merged.head())


## 4. 📊 Estatística Descritiva

In [ ]:
print("=" * 55)
print("📈 ESTATÍSTICA DESCRITIVA — FOCOS DE QUEIMADA")
print("=" * 55)
desc = df_queimadas_limpo['focos'].describe()
print(f"  Média          : {desc['mean']:,.1f} focos/mês")
print(f"  Mediana        : {df_queimadas_limpo['focos'].median():,.1f}")
print(f"  Desvio Padrão  : {desc['std']:,.1f}")
print(f"  Mínimo         : {desc['min']:,.0f}")
print(f"  Máximo         : {desc['max']:,.0f}")
print(f"  Assimetria     : {df_queimadas_limpo['focos'].skew():.3f}")
print(f"  Curtose        : {df_queimadas_limpo['focos'].kurt():.3f}")

print("\n📈 ESTATÍSTICA POR BIOMA:")
display(df_queimadas_limpo.groupby('bioma')['focos'].agg(
    ['mean','median','std','min','max']
).round(1).rename(columns={
    'mean':'Média','median':'Mediana',
    'std':'Desv. Padrão','min':'Mínimo','max':'Máximo'
}))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribuição das Variáveis Principais', fontsize=16, fontweight='bold', y=1.01)

# Histograma focos
axes[0].hist(df_queimadas_limpo['focos'], bins=40, color='#e07b39', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribuição — Focos de Queimada')
axes[0].set_xlabel('Focos / mês')
axes[0].set_ylabel('Frequência')
axes[0].axvline(df_queimadas_limpo['focos'].mean(), color='red', linestyle='--', label=f"Média: {df_queimadas_limpo['focos'].mean():.0f}")
axes[0].legend()

# Histograma precipitação
axes[1].hist(df_prec_limpo['precipitacao_mm'], bins=40, color='#4a90d9', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribuição — Precipitação (mm)')
axes[1].set_xlabel('Precipitação (mm)')
axes[1].set_ylabel('Frequência')
axes[1].axvline(df_prec_limpo['precipitacao_mm'].mean(), color='red', linestyle='--', label=f"Média: {df_prec_limpo['precipitacao_mm'].mean():.1f} mm")
axes[1].legend()

# Histograma NDVI
axes[2].hist(df_ndvi_limpo['ndvi_medio'], bins=30, color='#4caf50', edgecolor='white', alpha=0.85)
axes[2].set_title('Distribuição — NDVI Médio')
axes[2].set_xlabel('NDVI')
axes[2].set_ylabel('Frequência')
axes[2].axvline(df_ndvi_limpo['ndvi_medio'].mean(), color='red', linestyle='--', label=f"Média: {df_ndvi_limpo['ndvi_medio'].mean():.3f}")
axes[2].legend()

plt.tight_layout()
plt.savefig('distribuicoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Interpretação: Os focos de queimada apresentam distribuição assimétrica à direita,")
print("   indicando eventos extremos concentrados em períodos específicos (meses secos).")
print("   A precipitação e o NDVI tendem a distribuições mais simétricas por bioma.")


## 5. 📅 Análise de Série Temporal

In [ ]:
# Agrupamento por ano
anual = df_queimadas_limpo.groupby('ano')['focos'].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Evolução anual total
axes[0].plot(anual['ano'], anual['focos'], marker='o', linewidth=2.5,
             color='#e07b39', markersize=8)
axes[0].fill_between(anual['ano'], anual['focos'], alpha=0.15, color='#e07b39')
axes[0].set_title('Evolução Anual — Total de Focos no Brasil (2015–2024)')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Total de Focos')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for _, row in anual.iterrows():
    axes[0].annotate(f"{row['focos']:,.0f}", (row['ano'], row['focos']),
                     textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9)

# Sazonalidade mensal
mensal = df_queimadas_limpo.groupby('mes')['focos'].mean().reset_index()
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
cores = ['#e07b39' if m in [7,8,9,10] else '#4a90d9' for m in mensal['mes']]
axes[1].bar(meses, mensal['focos'], color=cores, edgecolor='white', alpha=0.85)
axes[1].set_title('Sazonalidade Mensal — Média de Focos (2015–2024)')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('Média de Focos')
axes[1].legend(handles=[
    plt.Rectangle((0,0),1,1, color='#e07b39', alpha=0.85, label='Período crítico (Jul–Out)'),
    plt.Rectangle((0,0),1,1, color='#4a90d9', alpha=0.85, label='Período ameno')
])

plt.tight_layout()
plt.savefig('serie_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Interpretação: O período de julho a outubro concentra os maiores volumes de queimadas,")
print("   correspondendo à estação seca no Brasil Central e Amazônia. Esse padrão sazonal")
print("   é fundamental para o planejamento de ações preventivas no sistema AgroSat.")


In [ ]:
# Evolução por bioma ao longo dos anos
bioma_ano = df_queimadas_limpo.groupby(['ano', 'bioma'])['focos'].sum().reset_index()
biomas_principais = ['Amazônia', 'Cerrado', 'Caatinga', 'Mata Atlântica']

fig, ax = plt.subplots(figsize=(14, 6))
cores_bioma = {'Amazônia': '#1a5f7a', 'Cerrado': '#c4a35a',
               'Caatinga': '#d4845a', 'Mata Atlântica': '#4caf50', 'Pampa': '#9c6b9e'}

for bioma in biomas_principais:
    dados = bioma_ano[bioma_ano['bioma'] == bioma]
    ax.plot(dados['ano'], dados['focos'], marker='o', linewidth=2,
            label=bioma, color=cores_bioma.get(bioma, '#888'))

ax.set_title('Evolução de Focos de Queimada por Bioma (2015–2024)')
ax.set_xlabel('Ano')
ax.set_ylabel('Total de Focos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(loc='upper left')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('focos_bioma.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Interpretação: Amazônia e Cerrado concentram historicamente os maiores volumes")
print("   de queimadas, sendo essas as regiões prioritárias para alertas do AgroSat.")


## 6. 🔗 Análise de Correlação

In [ ]:
# Correlação entre precipitação e focos de queimada
agg = df_merged.groupby(['ano', 'mes', 'estado']).agg(
    focos=('focos', 'sum'),
    precipitacao_mm=('precipitacao_mm', 'mean')
).reset_index()

pearson_r, pearson_p = pearsonr(agg['precipitacao_mm'], agg['focos'])
spearman_r, spearman_p = spearmanr(agg['precipitacao_mm'], agg['focos'])

print("=" * 55)
print("🔗 CORRELAÇÃO: Precipitação × Focos de Queimada")
print("=" * 55)
print(f"  Pearson  r = {pearson_r:.4f}  |  p-valor = {pearson_p:.4e}")
print(f"  Spearman r = {spearman_r:.4f}  |  p-valor = {spearman_p:.4e}")

sig = "✅ Estatisticamente significativa (p < 0,05)" if pearson_p < 0.05 else "❌ Não significativa"
print(f"\n  Interpretação: {sig}")
print(f"  Correlação {'negativa' if pearson_r < 0 else 'positiva'} de magnitude {'fraca' if abs(pearson_r) < 0.3 else 'moderada' if abs(pearson_r) < 0.6 else 'forte'}.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter
sample = agg.sample(min(800, len(agg)), random_state=42)
axes[0].scatter(sample['precipitacao_mm'], sample['focos'],
                alpha=0.4, color='#e07b39', s=20)
z = np.polyfit(agg['precipitacao_mm'], agg['focos'], 1)
p = np.poly1d(z)
x_range = np.linspace(agg['precipitacao_mm'].min(), agg['precipitacao_mm'].max(), 200)
axes[0].plot(x_range, p(x_range), 'r--', linewidth=2, label=f'Tendência (r={pearson_r:.3f})')
axes[0].set_title('Precipitação × Focos de Queimada')
axes[0].set_xlabel('Precipitação (mm)')
axes[0].set_ylabel('Focos de Queimada')
axes[0].legend()

# Matriz de correlação
corr_data = agg[['focos', 'precipitacao_mm']].rename(
    columns={'focos': 'Focos', 'precipitacao_mm': 'Precipitação'})
corr_data['Mês'] = agg['mes']

corr_matrix = corr_data.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, ax=axes[1], linewidths=0.5,
            annot_kws={'size': 13, 'weight': 'bold'})
axes[1].set_title('Matriz de Correlação')

plt.tight_layout()
plt.savefig('correlacao.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Interpretação: A correlação negativa confirma que maiores precipitações estão")
print("   associadas a menores focos de queimada — dado essencial para o modelo preditivo")
print("   do AgroSat identificar regiões em risco durante períodos de seca.")


## 7. 🧪 Teste de Hipóteses
**H₀:** Não há diferença significativa nos focos de queimada entre a estação seca (Jul–Out) e a estação chuvosa (Nov–Mar).  
**H₁:** A estação seca apresenta significativamente mais focos de queimada que a estação chuvosa.


In [ ]:
# Classificação dos meses em estações
def estacao(mes):
    if mes in [7, 8, 9, 10]:
        return 'Seca'
    elif mes in [11, 12, 1, 2, 3]:
        return 'Chuvosa'
    else:
        return 'Transição'

df_queimadas_limpo['estacao'] = df_queimadas_limpo['mes'].apply(estacao)

seca    = df_queimadas_limpo[df_queimadas_limpo['estacao'] == 'Seca']['focos']
chuvosa = df_queimadas_limpo[df_queimadas_limpo['estacao'] == 'Chuvosa']['focos']
transic = df_queimadas_limpo[df_queimadas_limpo['estacao'] == 'Transição']['focos']

# Teste t de Student (seca vs chuvosa)
t_stat, p_val = ttest_ind(seca, chuvosa, equal_var=False)

# ANOVA (3 grupos)
f_stat, p_anova = f_oneway(seca, chuvosa, transic)

print("=" * 55)
print("🧪 RESULTADO DOS TESTES DE HIPÓTESE")
print("=" * 55)
print(f"\n📊 Teste T (Seca vs Chuvosa):")
print(f"   t-estatístico : {t_stat:.4f}")
print(f"   p-valor       : {p_val:.2e}")
print(f"   Média Seca    : {seca.mean():,.1f} focos/mês")
print(f"   Média Chuvosa : {chuvosa.mean():,.1f} focos/mês")
print(f"   Diferença     : {((seca.mean()-chuvosa.mean())/chuvosa.mean()*100):.1f}% maior na seca")

if p_val < 0.05:
    print("\n  ✅ Rejeitamos H₀: há diferença estatisticamente significativa.")
    print("     A estação seca apresenta mais focos que a chuvosa (p < 0,05).")
else:
    print("\n  ❌ Não rejeitamos H₀.")

print(f"\n📊 ANOVA (3 estações):")
print(f"   F-estatístico : {f_stat:.4f}")
print(f"   p-valor       : {p_anova:.2e}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot por estação
ordem = ['Seca', 'Transição', 'Chuvosa']
cores_estacao = {'Seca': '#e07b39', 'Transição': '#f5c842', 'Chuvosa': '#4a90d9'}
data_box = df_queimadas_limpo[df_queimadas_limpo['estacao'].isin(ordem)]

sns.boxplot(data=data_box, x='estacao', y='focos', order=ordem,
            palette=cores_estacao, ax=axes[0])
axes[0].set_title(f'Focos de Queimada por Estação\n(p-valor Teste T = {p_val:.2e})')
axes[0].set_xlabel('Estação')
axes[0].set_ylabel('Focos de Queimada / mês')

# Violinplot por bioma
biomas_plot = ['Amazônia', 'Cerrado', 'Caatinga']
data_viol = df_queimadas_limpo[df_queimadas_limpo['bioma'].isin(biomas_plot)]
sns.violinplot(data=data_viol, x='bioma', y='focos',
               palette='Set2', ax=axes[1], inner='box')
axes[1].set_title('Distribuição de Focos por Bioma (Violinplot)')
axes[1].set_xlabel('Bioma')
axes[1].set_ylabel('Focos de Queimada / mês')

plt.tight_layout()
plt.savefig('teste_hipotese.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Interpretação: O teste confirma que a estação seca gera significativamente")
print("   mais queimadas. Isso valida o uso de alertas sazonais no AgroSat,")
print("   notificando produtores com antecedência nos meses de maior risco.")


## 8. 🌿 Análise NDVI — Saúde da Vegetação

In [ ]:
# NDVI médio por bioma e ano
ndvi_bioma_ano = df_ndvi_limpo.groupby(['ano', 'bioma'])['ndvi_medio'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Tendência NDVI por bioma
biomas_ndvi = df_ndvi_limpo['bioma'].unique()
for bioma in biomas_ndvi:
    dados = ndvi_bioma_ano[ndvi_bioma_ano['bioma'] == bioma]
    axes[0].plot(dados['ano'], dados['ndvi_medio'], marker='s', linewidth=2,
                 label=bioma, color=cores_bioma.get(bioma, '#888'))

axes[0].set_title('Evolução do NDVI Médio por Bioma (2015–2024)')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('NDVI Médio')
axes[0].legend()
axes[0].axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='Limiar vegetação saudável')

# Ranking de NDVI médio por bioma
ndvi_rank = df_ndvi_limpo.groupby('bioma')['ndvi_medio'].mean().sort_values(ascending=True)
cores_rank = ['#c0392b' if v < 0.4 else '#f39c12' if v < 0.6 else '#27ae60'
              for v in ndvi_rank.values]
axes[1].barh(ndvi_rank.index, ndvi_rank.values, color=cores_rank, edgecolor='white', alpha=0.85)
axes[1].set_title('NDVI Médio por Bioma (2015–2024)')
axes[1].set_xlabel('NDVI Médio')
for i, (bioma, val) in enumerate(ndvi_rank.items()):
    axes[1].text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=11)

axes[1].axvline(0.5, color='gray', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.savefig('ndvi_analise.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Interpretação: O NDVI (Índice de Vegetação por Diferença Normalizada) indica")
print("   a saúde e densidade da cobertura vegetal. Valores acima de 0,5 indicam")
print("   vegetação densa e saudável. O AgroSat usa esse índice para identificar")
print("   áreas com cobertura vegetal comprometida, auxiliando no plantio e recuperação.")


## 9. 📐 Taxa de Variação — Limites e Derivadas

In [ ]:
# Taxa de variação anual dos focos (derivada discreta)
anual_total = df_queimadas_limpo.groupby('ano')['focos'].sum().reset_index()
anual_total['variacao_abs'] = anual_total['focos'].diff()
anual_total['variacao_pct'] = anual_total['focos'].pct_change() * 100

print("=" * 55)
print("📐 TAXA DE VARIAÇÃO ANUAL — FOCOS DE QUEIMADA")
print("=" * 55)
display(anual_total.dropna().rename(columns={
    'ano': 'Ano', 'focos': 'Total Focos',
    'variacao_abs': 'Δ Absoluto', 'variacao_pct': 'Δ % Anual'
}).round(1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Total anual
axes[0].bar(anual_total['ano'], anual_total['focos'],
            color='#e07b39', alpha=0.8, edgecolor='white')
axes[0].plot(anual_total['ano'], anual_total['focos'],
             color='#333', marker='o', linewidth=2)
axes[0].set_title('Total Anual de Focos')
axes[0].set_ylabel('Focos')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Taxa de variação %
cores_var = ['#27ae60' if v <= 0 else '#e74c3c'
             for v in anual_total['variacao_pct'].fillna(0)]
axes[1].bar(anual_total['ano'], anual_total['variacao_pct'].fillna(0),
            color=cores_var, alpha=0.85, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Taxa de Variação Anual dos Focos (%)')
axes[1].set_ylabel('Variação (%)')

plt.tight_layout()
plt.savefig('taxa_variacao.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Interpretação: A taxa de variação (derivada discreta) revela os anos com maior")
print("   aceleração nos focos. Picos positivos indicam anos críticos que demandam")
print("   atenção redobrada — informação central para os alertas preventivos do AgroSat.")


## 10. ∫ Focos Acumulados — Conceito de Integral

In [ ]:
# Acumulado ao longo do período (conceito de integral discreta)
anual_total['focos_acumulado'] = anual_total['focos'].cumsum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Acumulado
axes[0].fill_between(anual_total['ano'], anual_total['focos_acumulado'],
                     alpha=0.3, color='#e07b39')
axes[0].plot(anual_total['ano'], anual_total['focos_acumulado'],
             color='#e07b39', marker='o', linewidth=2.5)
axes[0].set_title('Focos Acumulados no Brasil (2015–2024)')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Focos Acumulados')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for _, row in anual_total.iterrows():
    axes[0].annotate(f"{row['focos_acumulado']/1e6:.1f}M",
                     (row['ano'], row['focos_acumulado']),
                     textcoords="offset points", xytext=(0, 8), ha='center', fontsize=8)

# Por região
regiao_ano = df_queimadas_limpo.groupby(['ano','regiao'])['focos'].sum().reset_index()
for regiao in regiao_ano['regiao'].unique():
    d = regiao_ano[regiao_ano['regiao']==regiao].sort_values('ano')
    d = d.copy()
    d['acum'] = d['focos'].cumsum()
    axes[1].plot(d['ano'], d['acum'], marker='o', linewidth=2, label=regiao)

axes[1].set_title('Focos Acumulados por Região')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Focos Acumulados')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[1].legend()

plt.tight_layout()
plt.savefig('acumulado.png', dpi=150, bbox_inches='tight')
plt.show()

total_acum = anual_total['focos_acumulado'].iloc[-1]
print(f"\n📌 Total acumulado (2015–2024): {total_acum:,.0f} focos de queimada no Brasil.")
print("   A integral discreta mostra o impacto cumulativo ao longo do período,")
print("   reforçando a urgência de soluções de monitoramento como o AgroSat.")


## 11. 🗺️ Dashboard Executivo — Protótipo AgroSat

In [ ]:
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#1a1a2e')

ax_title = fig.add_axes([0.0, 0.93, 1.0, 0.07])
ax_title.set_facecolor('#16213e')
ax_title.text(0.5, 0.5, '🛰️  AgroSat — Dashboard de Monitoramento Agrícola por Satélite',
              transform=ax_title.transAxes, ha='center', va='center',
              fontsize=18, color='white', fontweight='bold')
ax_title.axis('off')

ax1 = fig.add_axes([0.05, 0.62, 0.40, 0.27])
ax2 = fig.add_axes([0.55, 0.62, 0.40, 0.27])
ax3 = fig.add_axes([0.05, 0.30, 0.40, 0.27])
ax4 = fig.add_axes([0.55, 0.30, 0.40, 0.27])
ax5 = fig.add_axes([0.05, 0.05, 0.88, 0.20])

for ax in [ax1, ax2, ax3, ax4, ax5]:
    ax.set_facecolor('#16213e')
    for spine in ax.spines.values():
        spine.set_edgecolor('#0f3460')

# 1) Série anual
ax1.plot(anual_total['ano'], anual_total['focos'], marker='o',
         color='#e07b39', linewidth=2.5)
ax1.fill_between(anual_total['ano'], anual_total['focos'], alpha=0.2, color='#e07b39')
ax1.set_title('Focos Anuais — Brasil', color='white', fontsize=12)
ax1.tick_params(colors='white')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax1.set_facecolor('#16213e')

# 2) Sazonalidade
ax2.bar(meses, mensal['focos'],
        color=['#e07b39' if m in [7,8,9,10] else '#4a90d9' for m in range(1,13)],
        alpha=0.85, edgecolor='#1a1a2e')
ax2.set_title('Sazonalidade Mensal', color='white', fontsize=12)
ax2.tick_params(colors='white', axis='both')
ax2.set_facecolor('#16213e')

# 3) NDVI por bioma
ndvi_bar = df_ndvi_limpo.groupby('bioma')['ndvi_medio'].mean().sort_values()
cores_ndvi = ['#c0392b' if v < 0.4 else '#f39c12' if v < 0.6 else '#27ae60'
              for v in ndvi_bar.values]
ax3.barh(ndvi_bar.index, ndvi_bar.values, color=cores_ndvi, alpha=0.85)
ax3.set_title('NDVI Médio por Bioma', color='white', fontsize=12)
ax3.tick_params(colors='white')
ax3.set_facecolor('#16213e')

# 4) Correlação scatter
ax4.scatter(agg['precipitacao_mm'], agg['focos'],
            alpha=0.3, color='#4a90d9', s=8)
ax4.set_title(f'Precipitação × Focos (r={pearson_r:.2f})', color='white', fontsize=12)
ax4.tick_params(colors='white')
ax4.set_facecolor('#16213e')

# 5) Ranking top 10 estados
top10 = (df_queimadas_limpo.groupby('estado')['focos'].sum()
         .sort_values(ascending=False).head(10))
ax5.bar(top10.index, top10.values, color='#e07b39', alpha=0.85, edgecolor='#1a1a2e')
ax5.set_title('Top 10 Estados com Mais Focos (2015–2024)', color='white', fontsize=12)
ax5.tick_params(colors='white')
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax5.set_facecolor('#16213e')

plt.savefig('dashboard_agrosat.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print("\n✅ Dashboard executivo do AgroSat gerado com sucesso!")


## 12. ✅ Conclusão e Contribuição para o Global Solution

### 🎯 Síntese dos Resultados

| Análise | Resultado Principal |
|---------|-------------------|
| Estatística Descritiva | Focos concentrados no bioma Amazônia e Cerrado |
| Série Temporal | Pico sazonal de Jul–Out; tendência de crescimento 2015–2021 |
| Correlação | Precipitação e focos têm correlação negativa significativa |
| Teste de Hipótese | Estação seca tem significativamente mais queimadas (p < 0,05) |
| NDVI | Amazônia com maior cobertura vegetal; Caatinga com maior vulnerabilidade |
| Taxa de Variação | Identificação dos anos mais críticos para ação preventiva |
| Focos Acumulados | Impacto cumulativo reforça urgência de monitoramento contínuo |

### 🛰️ Como a análise contribui para o AgroSat

1. **Alertas sazonais:** A sazonalidade confirmada (Jul–Out) permite programar alertas preventivos com antecedência para produtores rurais.

2. **Priorização geográfica:** Os estados e biomas com maior incidência histórica recebem monitoramento intensificado no sistema.

3. **Modelo preditivo:** A correlação entre precipitação e focos embasa um modelo de risco que cruza previsão climática com padrões históricos.

4. **Índice de saúde vegetal:** O NDVI permite ao AgroSat identificar áreas com cobertura comprometida antes que o agricultor perceba visualmente.

5. **Base de evidências:** Os dados analisados fundamentam políticas públicas e decisões de cooperativas agrícolas.

### 🌍 Impacto nos ODS
- **ODS 2:** Alertas ajudam a evitar perdas de lavoura por queimadas
- **ODS 13:** Monitoramento ambiental contribui para mitigação climática  
- **ODS 15:** Proteção dos biomas por meio de dados e inteligência artificial

---
*Projeto desenvolvido para a Global Solutions 2026 — FIAP*  
*Dados gerados com base na metodologia INPE/BDQueimadas e NASA EarthData*
